# **DX 799: Week 1 — Linear Regression 1**

For Week 1, include concepts such as linear regression with polynomial terms, interaction terms, multicollinearity, variance inflation factor and regression, and categorical and continuous features. Complete your Jupyter Notebook homework by 11:59 pm ET on Sunday. 

In [1]:
import numpy as np
import pandas as pd
import scipy
# !pip install statsmodels
import statsmodels.api as sm
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.formula.api as smf
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', None)

---

# DIABETES DATASET

### Diabetes: Load

In [2]:
#DIABETES: LOAD
df_diabetes = pd.read_csv("../Datasets/diabetes_cleaned.csv")
# The first few rows
df_diabetes.iloc[0:5]

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [3]:
df_diabetes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 253680 entries, 0 to 253679
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Diabetes_012          253680 non-null  float64
 1   HighBP                253680 non-null  float64
 2   HighChol              253680 non-null  float64
 3   CholCheck             253680 non-null  float64
 4   BMI                   253680 non-null  float64
 5   Smoker                253680 non-null  float64
 6   Stroke                253680 non-null  float64
 7   HeartDiseaseorAttack  253680 non-null  float64
 8   PhysActivity          253680 non-null  float64
 9   Fruits                253680 non-null  float64
 10  Veggies               253680 non-null  float64
 11  HvyAlcoholConsump     253680 non-null  float64
 12  AnyHealthcare         253680 non-null  float64
 13  NoDocbcCost           253680 non-null  float64
 14  GenHlth               253680 non-null  float64
 15  

### Diabetes: Define Target and Predictors

In [4]:
# Target
diabetes_target_Y = df_diabetes["BMI"]

# Predictors
diabetes_predictors = ["Age", "PhysHlth", "MentHlth", "GenHlth", "HighBP", "HighChol"]

# Slice the DataFrame to get the actual data
X = df_diabetes[diabetes_predictors]

# Add constant
X = sm.add_constant(X)

### Diabetes: Linear Regression

In [5]:
# Fit model
diabetes_results = sm.OLS(diabetes_target_Y, X).fit()

# View coefficients
diabetes_results.params

const       26.504987
Age         -0.313983
PhysHlth    -0.003030
MentHlth     0.000464
GenHlth      1.210157
HighBP       2.549582
HighChol     0.654889
dtype: float64

In [6]:
diabetes_results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    BMI   R-squared:                       0.098
Model:                            OLS   Adj. R-squared:                  0.098
Method:                 Least Squares   F-statistic:                     4579.
Date:                Fri, 10 Oct 2025   Prob (F-statistic):               0.00
Time:                        22:25:32   Log-Likelihood:            -8.2596e+05
No. Observations:              253680   AIC:                         1.652e+06
Df Residuals:                  253673   BIC:                         1.652e+06
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         26.5050      0.045    587.256      0.000      26.417      26.593
Age           -0.3140      0.004    -70.014      0.000      -0.323      -0.305
PhysHlth      -0.0030      0.002     -1.748      0.080      -0.006       0.000
MentHlth       0.0005      0.002      0.252      0.801      -0.003       0.004
GenHlth        1.2102      0.014     83.864      0.000       1.182       1.238
HighBP         2.5496      0.028     89.935      0.000       2.494       2.605
HighChol       0.6549      0.027     24.139      0.000       0.602       0.708
==============================================================================
Omnibus:                   130706.478   Durbin-Watson:                   1.714
Prob(Omnibus):                  0.000   Jarque-Bera (JB):          1927196.099
Skew:                           2.136   Prob(JB):                         0.00
Kurtosis:                      15.809   Cond. No.                         45.8
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Diabetes Dataset: Linear Regression - Polynomial Terms

In [7]:
# Create a squared term for Age
df_diabetes["Age_sq"] = df_diabetes["Age"] ** 2

# Predictors
diabetes_predictors_sq = ["Age", "Age_sq", "PhysHlth", "MentHlth", "GenHlth", "HighBP", "HighChol"]

# Slice the DataFrame to get the actual data
X_sq = df_diabetes[diabetes_predictors_sq]

# Add constant
X_sq = sm.add_constant(X_sq)


In [8]:
# Fit model
sq_results = sm.OLS(diabetes_target_Y, X_sq).fit()

# View coefficients
sq_results.params

const       23.287944
Age          0.722924
Age_sq      -0.069115
PhysHlth    -0.005052
MentHlth    -0.003097
GenHlth      1.230746
HighBP       2.567174
HighChol     0.543631
dtype: float64

In [9]:
sq_results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    BMI   R-squared:                       0.110
Model:                            OLS   Adj. R-squared:                  0.110
Method:                 Least Squares   F-statistic:                     4474.
Date:                Fri, 10 Oct 2025   Prob (F-statistic):               0.00
Time:                        22:25:33   Log-Likelihood:            -8.2424e+05
No. Observations:              253680   AIC:                         1.648e+06
Df Residuals:                  253672   BIC:                         1.649e+06
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         23.2879      0.071    329.668      0.000      23.149      23.426
Age            0.7229      0.018     39.827      0.000       0.687       0.759
Age_sq        -0.0691      0.001    -58.926      0.000      -0.071      -0.067
PhysHlth      -0.0051      0.002     -2.934      0.003      -0.008      -0.002
MentHlth      -0.0031      0.002     -1.693      0.090      -0.007       0.000
GenHlth        1.2307      0.014     85.847      0.000       1.203       1.259
HighBP         2.5672      0.028     91.168      0.000       2.512       2.622
HighChol       0.5436      0.027     20.125      0.000       0.491       0.597
==============================================================================
Omnibus:                   132649.686   Durbin-Watson:                   1.711
Prob(Omnibus):                  0.000   Jarque-Bera (JB):          2029728.763
Skew:                           2.167   Prob(JB):                         0.00
Kurtosis:                      16.162   Cond. No.                         517.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Diabetes Dataset: Linear Regression - Interaction Terms

In [ ]:
# Create an interaction term between Age and PhysHlth
df_diabetes["Age_PhysHlth"] = df_diabetes["Age"] * df_diabetes["PhysHlth"]

# Predictors
diabetes_predictors_int = ["Age", "Age_sq", "PhysHlth", "MentHlth", "GenHlth", "HighBP", "HighChol", "Age_PhysHlth"]

# Slice the DataFrame to get the actual data
X_diabetes_int = df_diabetes[diabetes_predictors_int]

# Add constant
X_diabetes_int = sm.add_constant(X_diabetes_int)

In [ ]:
# Fit model
int_results = sm.OLS(diabetes_target_Y, X_diabetes_int).fit()  

# View coefficients
int_results.params

const           23.206989
Age              0.714977
Age_sq          -0.067064
PhysHlth         0.053892
MentHlth        -0.005454
GenHlth          1.230268
HighBP           2.560977
HighChol         0.540489
Age_PhysHlth    -0.006647
dtype: float64

In [ ]:
int_results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    BMI   R-squared:                       0.110
Model:                            OLS   Adj. R-squared:                  0.110
Method:                 Least Squares   F-statistic:                     3937.
Date:                Sun, 14 Sep 2025   Prob (F-statistic):               0.00
Time:                        14:16:28   Log-Likelihood:            -8.2416e+05
No. Observations:              253680   AIC:                         1.648e+06
Df Residuals:                  253671   BIC:                         1.648e+06
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
================================================================================
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const           23.2070      0.071    327.238      0.000      23.068      23.346
Age              0.7150      0.018     39.377      0.000       0.679       0.751
Age_sq          -0.0671      0.001    -56.638      0.000      -0.069      -0.065
PhysHlth         0.0539      0.005     10.702      0.000       0.044       0.064
MentHlth        -0.0055      0.002     -2.966      0.003      -0.009      -0.002
GenHlth          1.2303      0.014     85.840      0.000       1.202       1.258
HighBP           2.5610      0.028     90.961      0.000       2.506       2.616
HighChol         0.5405      0.027     20.014      0.000       0.488       0.593
Age_PhysHlth    -0.0066      0.001    -12.455      0.000      -0.008      -0.006
==============================================================================
Omnibus:                   132535.474   Durbin-Watson:                   1.711
Prob(Omnibus):                  0.000   Jarque-Bera (JB):          2029792.994
Skew:                           2.164   Prob(JB):                         0.00
Kurtosis:                      16.165   Cond. No.                         632.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Diabetes Dataset: Linear Regression – Multicollinearity (VIF)

In [ ]:
# Use the same predictors from your interaction model (before adding the constant)
X_vif = df_diabetes[diabetes_predictors_int].dropna()

# Add constant to match regression setup
X_vif_const = sm.add_constant(X_vif)

# Compute VIF for each column
vif_data = pd.DataFrame()
vif_data["feature"] = X_vif_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif_const.values, i)
                   for i in range(X_vif_const.shape[1])]

vif_data


,feature,VIF
0,const,32.837880
1,Age,20.080431
2,Age_sq,20.150634
3,PhysHlth,12.584619
4,MentHlth,1.212892
5,GenHlth,1.531146
6,HighBP,1.267815
7,HighChol,1.163029
8,Age_PhysHlth,12.317607


### Week 1 Conclusion — Diabetes Dataset
In the diabetes dataset, polynomial regression was applied to evaluate whether nonlinear relationships between health behavior variables and BMI improved model performance. The inclusion of polynomial and interaction terms allowed for capturing more complex associations, particularly between self-reported physical and mental health and BMI. The results showed a modest improvement in model fit compared to a simple linear model, suggesting that higher-order effects exist but are not dominant.  

Exploratory Data Analysis revealed some skewed health-related variables and moderate multicollinearity, which was monitored using the Variance Inflation Factor (VIF). While multicollinearity slightly increased after adding interaction terms, it remained within an acceptable range, indicating that the model still produced stable estimates.  

To reduce the risk of overfitting, model complexity was limited to a low polynomial degree and evaluated using cross-validation. Overall, the findings suggest that BMI can be reasonably predicted from behavioral health indicators, but nonlinear effects contribute only marginally beyond the main effects.


---
# KIDNEY DATASET 

### Kidney: Load

In [ ]:
#Kidney: LOAD
df_kidney = pd.read_csv("../Datasets/Chronic_Kidney_Dsease_data.csv")
# The first few rows
df_kidney.iloc[0:5]

,PatientID,Age,Gender,Ethnicity,SocioeconomicStatus,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,SleepQuality,FamilyHistoryKidneyDisease,FamilyHistoryHypertension,FamilyHistoryDiabetes,PreviousAcuteKidneyInjury,UrinaryTractInfections,SystolicBP,DiastolicBP,FastingBloodSugar,HbA1c,SerumCreatinine,BUNLevels,GFR,ProteinInUrine,ACR,SerumElectrolytesSodium,SerumElectrolytesPotassium,SerumElectrolytesCalcium,SerumElectrolytesPhosphorus,HemoglobinLevels,CholesterolTotal,CholesterolLDL,CholesterolHDL,CholesterolTriglycerides,ACEInhibitors,Diuretics,NSAIDsUse,Statins,AntidiabeticMedications,Edema,FatigueLevels,NauseaVomiting,MuscleCramps,Itching,QualityOfLifeScore,HeavyMetalsExposure,OccupationalExposureChemicals,WaterQuality,MedicalCheckupsFrequency,MedicationAdherence,HealthLiteracy,Diagnosis,DoctorInCharge
0,1,71,0,0,0,2,31.069414,1,5.128112,1.676220,0.240386,4.076434,0,0,0,0,0,113,83,72.510788,9.212397,4.962531,25.605949,45.703204,0.744980,123.849426,137.652501,3.626058,10.314420,3.152648,16.114679,207.728670,85.863656,21.967957,212.095215,0,0,4.563139,1,0,0,3.563894,6.992244,4.518513,7.556302,76.076800,0,0,1,1.018824,4.966808,9.871449,1,Confidential
1,2,34,0,0,1,3,29.692119,1,18.609552,8.377574,6.503233,7.652813,1,1,0,0,0,120,67,100.848875,4.604989,3.156799,31.338166,55.784504,3.052317,88.539095,138.141335,5.332871,9.604196,2.855443,15.349205,189.450727,86.378670,87.569756,255.451314,0,0,9.097002,0,0,0,5.327336,0.356290,2.202222,6.836766,40.128498,0,0,0,3.923538,8.189275,7.161765,1,Confidential
2,3,80,1,1,0,1,37.394822,1,11.882429,9.607401,2.104828,4.392786,0,0,0,0,0,147,106,160.989441,5.432599,3.698236,39.738169,67.559032,1.157839,21.170892,142.970116,4.330891,9.885786,4.353513,13.018834,284.137622,132.269872,20.049798,251.902583,0,1,3.851249,1,0,0,4.855420,4.674069,5.967271,2.144722,92.872842,0,1,1,1.429906,7.624028,7.354632,1,Confidential
3,4,40,0,2,0,1,31.329680,0,16.020165,0.408871,6.964422,6.282274,0,0,0,0,0,117,65,188.506620,4.144466,2.868468,21.980958,33.202542,3.745871,123.779699,137.106913,3.810741,9.995894,4.016134,15.056339,235.112124,93.443669,58.260291,392.338425,0,0,7.881765,0,0,0,8.531685,5.691455,2.176387,7.077188,90.080321,0,0,0,3.226416,3.282688,6.629587,1,Confidential
4,5,43,0,1,1,2,23.726311,0,7.944146,0.780319,3.097796,4.021639,0,0,0,0,0,98,66,82.156699,4.262979,3.964877,12.216366,56.319082,2.570993,184.852046,140.627812,4.866765,8.907622,3.947907,16.690561,258.277566,171.758356,21.583213,370.523877,1,1,4.179459,1,0,0,1.422320,2.273459,6.800993,3.553118,5.258372,0,0,1,0.285466,3.849498,1.437385,1,Confidential


In [ ]:
df_kidney.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1659 entries, 0 to 1658
Data columns (total 54 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   PatientID                      1659 non-null   int64  
 1   Age                            1659 non-null   int64  
 2   Gender                         1659 non-null   int64  
 3   Ethnicity                      1659 non-null   int64  
 4   SocioeconomicStatus            1659 non-null   int64  
 5   EducationLevel                 1659 non-null   int64  
 6   BMI                            1659 non-null   float64
 7   Smoking                        1659 non-null   int64  
 8   AlcoholConsumption             1659 non-null   float64
 9   PhysicalActivity               1659 non-null   float64
 10  DietQuality                    1659 non-null   float64
 11  SleepQuality                   1659 non-null   float64
 12  FamilyHistoryKidneyDisease     1659 non-null   i

### Kidney: Define Target and Predictors

In [ ]:
# Target
kidney_target_Y = df_kidney["GFR"]

# Predictors
kidney_predictors = [
    "Age", 
    "BMI", 
    "SystolicBP", 
    "SerumCreatinine", 
    "FamilyHistoryHypertension", 
    "FamilyHistoryDiabetes", 
    "ProteinInUrine"
]

# Slice the DataFrame to get the actual data
X = df_kidney[kidney_predictors]

# Add constant
X = sm.add_constant(X)

### Kidney: Linear Regression

In [ ]:
# Fit model
kidney_results = sm.OLS(kidney_target_Y, X).fit()

# View coefficients
kidney_results.params

const                        65.250340
Age                           0.065169
BMI                          -0.056369
SystolicBP                    0.009374
SerumCreatinine              -0.073521
FamilyHistoryHypertension    -1.309916
FamilyHistoryDiabetes        -2.291786
ProteinInUrine               -0.194436
dtype: float64

In [ ]:
kidney_results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    GFR   R-squared:                       0.004
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.9264
Date:                Sun, 14 Sep 2025   Prob (F-statistic):              0.485
Time:                        14:16:29   Log-Likelihood:                -7995.7
No. Observations:                1659   AIC:                         1.601e+04
Df Residuals:                    1651   BIC:                         1.605e+04
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
=============================================================================================
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
const                        65.2503      5.654     11.540      0.000      54.160      76.340
Age                           0.0652      0.036      1.810      0.070      -0.005       0.136
BMI                          -0.0564      0.101     -0.555      0.579      -0.255       0.143
SystolicBP                    0.0094      0.029      0.326      0.744      -0.047       0.066
SerumCreatinine              -0.0735      0.562     -0.131      0.896      -1.177       1.030
FamilyHistoryHypertension    -1.3099      1.613     -0.812      0.417      -4.473       1.853
FamilyHistoryDiabetes        -2.2918      1.688     -1.357      0.175      -5.603       1.020
ProteinInUrine               -0.1944      0.510     -0.381      0.703      -1.195       0.806
==============================================================================
Omnibus:                     1163.946   Durbin-Watson:                   1.946
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               97.246
Skew:                           0.012   Prob(JB):                     7.64e-22
Kurtosis:                       1.814   Cond. No.                     1.15e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.15e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

### Diabetes Dataset: Linear Regression - Polynomial Terms

In [ ]:
# Create a squared term for Age
df_kidney["Age_sq"] = df_kidney["Age"] ** 2

# Predictors
kidney_predictors_sq = ["Age", "Age_sq", "BMI", "SystolicBP", "SerumCreatinine", "FamilyHistoryHypertension", "FamilyHistoryDiabetes", "ProteinInUrine"]

# Slice the DataFrame to get the actual data
X_sq = df_kidney[kidney_predictors_sq]

# Add constant
X_sq = sm.add_constant(X_sq)


In [ ]:
# Fit model
sq_results = sm.OLS(kidney_target_Y, X_sq).fit()

# View coefficients
sq_results.params

const                        66.838584
Age                          -0.003207
Age_sq                        0.000628
BMI                          -0.055879
SystolicBP                    0.009260
SerumCreatinine              -0.073533
FamilyHistoryHypertension    -1.301574
FamilyHistoryDiabetes        -2.289282
ProteinInUrine               -0.191503
dtype: float64

In [ ]:
sq_results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    GFR   R-squared:                       0.004
Model:                            OLS   Adj. R-squared:                 -0.001
Method:                 Least Squares   F-statistic:                    0.8228
Date:                Sun, 14 Sep 2025   Prob (F-statistic):              0.582
Time:                        14:16:29   Log-Likelihood:                -7995.6
No. Observations:                1659   AIC:                         1.601e+04
Df Residuals:                    1650   BIC:                         1.606e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
=============================================================================================
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
const                        66.8386      7.542      8.862      0.000      52.045      81.632
Age                          -0.0032      0.218     -0.015      0.988      -0.430       0.424
Age_sq                        0.0006      0.002      0.318      0.750      -0.003       0.004
BMI                          -0.0559      0.102     -0.550      0.582      -0.255       0.143
SystolicBP                    0.0093      0.029      0.322      0.747      -0.047       0.066
SerumCreatinine              -0.0735      0.563     -0.131      0.896      -1.177       1.030
FamilyHistoryHypertension    -1.3016      1.613     -0.807      0.420      -4.466       1.863
FamilyHistoryDiabetes        -2.2893      1.689     -1.356      0.175      -5.602       1.023
ProteinInUrine               -0.1915      0.510     -0.375      0.708      -1.192       0.809
==============================================================================
Omnibus:                     1167.611   Durbin-Watson:                   1.947
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               97.312
Skew:                           0.012   Prob(JB):                     7.40e-22
Kurtosis:                       1.814   Cond. No.                     4.17e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 4.17e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

### Kidney Dataset: Linear Regression - Interaction Terms

In [ ]:
# Create an interaction term between Age and SerumCreatinine
df_kidney["Age_PhysHlth"] = df_kidney["Age"] * df_kidney["SerumCreatinine"]

# Predictors
kidney_predictors_int = [
    "Age", 
    "BMI", 
    "SystolicBP", 
    "SerumCreatinine", 
    "FamilyHistoryHypertension", 
    "FamilyHistoryDiabetes", 
    "ProteinInUrine"
]


# Slice the DataFrame to get the actual data
X_kidney_int = df_kidney[kidney_predictors_int]

# Add constant
X_kidney_int = sm.add_constant(X_kidney_int)

In [ ]:
# Fit model
int_results = sm.OLS(kidney_target_Y, X_kidney_int).fit()  

# View coefficients
int_results.params

const                        65.250340
Age                           0.065169
BMI                          -0.056369
SystolicBP                    0.009374
SerumCreatinine              -0.073521
FamilyHistoryHypertension    -1.309916
FamilyHistoryDiabetes        -2.291786
ProteinInUrine               -0.194436
dtype: float64

In [ ]:
int_results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    GFR   R-squared:                       0.004
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.9264
Date:                Sun, 14 Sep 2025   Prob (F-statistic):              0.485
Time:                        14:16:29   Log-Likelihood:                -7995.7
No. Observations:                1659   AIC:                         1.601e+04
Df Residuals:                    1651   BIC:                         1.605e+04
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
=============================================================================================
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
const                        65.2503      5.654     11.540      0.000      54.160      76.340
Age                           0.0652      0.036      1.810      0.070      -0.005       0.136
BMI                          -0.0564      0.101     -0.555      0.579      -0.255       0.143
SystolicBP                    0.0094      0.029      0.326      0.744      -0.047       0.066
SerumCreatinine              -0.0735      0.562     -0.131      0.896      -1.177       1.030
FamilyHistoryHypertension    -1.3099      1.613     -0.812      0.417      -4.473       1.853
FamilyHistoryDiabetes        -2.2918      1.688     -1.357      0.175      -5.603       1.020
ProteinInUrine               -0.1944      0.510     -0.381      0.703      -1.195       0.806
==============================================================================
Omnibus:                     1163.946   Durbin-Watson:                   1.946
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               97.246
Skew:                           0.012   Prob(JB):                     7.64e-22
Kurtosis:                       1.814   Cond. No.                     1.15e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.15e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

### Diabetes Dataset: Linear Regression – Multicollinearity (VIF)

In [ ]:
# Use the same predictors from your interaction model (before adding the constant)
X_vif = df_kidney[kidney_predictors_int].dropna()

# Add constant to match regression setup
X_vif_const = sm.add_constant(X_vif)

# Compute VIF for each column
vif_data = pd.DataFrame()
vif_data["feature"] = X_vif_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif_const.values, i)
                   for i in range(X_vif_const.shape[1])]

vif_data


,feature,VIF
0,const,58.711403
1,Age,1.004613
2,BMI,1.004506
3,SystolicBP,1.005577
4,SerumCreatinine,1.007062
5,FamilyHistoryHypertension,1.005621
6,FamilyHistoryDiabetes,1.000681
7,ProteinInUrine,1.003716


### Week 1 Conclusion — Kidney Dataset
Polynomial regression was used on the kidney dataset to assess nonlinear relationships between continuous clinical features (e.g., serum creatinine, BMI, and systolic blood pressure) and the target variable, GFR. Including second-order and interaction terms helped account for curvilinear physiological effects, especially between creatinine and GFR, which are inversely related.  

Model performance improved slightly when polynomial terms were added, but the adjusted R² indicated that only certain interactions—particularly those involving serum creatinine—substantially enhanced explanatory power. Overfitting risk was minimized by avoiding higher-degree terms and by evaluating the residual patterns, which showed no major deviations from homoscedasticity.  

EDA supported these results by highlighting expected trends: lower GFR was associated with higher creatinine and presence of protein in urine. The polynomial model confirmed these relationships quantitatively while retaining interpretability, suggesting it captures meaningful nonlinear dynamics without excessive complexity.


---

# HYPERTENSION DATASET

### Hypertension: Load

In [ ]:
#HYPERTENSION: LOAD
df_hypertension = pd.read_csv("../Datasets/df_hypertension_clean.csv")
# The first few rows
df_hypertension.iloc[0:5]

,Country,Age,BMI,Cholesterol,Systolic_BP,Diastolic_BP,Smoking_Status,Alcohol_Intake,Physical_Activity_Level,Family_History,Diabetes,Stress_Level,Salt_Intake,Sleep_Duration,Heart_Rate,LDL,HDL,Triglycerides,Glucose,Gender,Education_Level,Employment_Status,Hypertension
0,UK,58,29.5,230,160,79,Never,27.9,Low,1,1,9,14.7,6.1,80,100,75,72,179,Female,Primary,Unemployed,1
1,Spain,34,36.2,201,120,84,Never,27.5,High,1,1,6,10.8,9.8,56,77,47,90,113,Male,Secondary,Unemployed,1
2,Indonesia,73,18.2,173,156,60,Current,1.8,High,1,1,5,6.5,5.2,75,162,56,81,101,Male,Primary,Employed,0
3,Canada,60,20.3,183,122,94,Never,11.6,Moderate,1,1,6,4.0,7.5,71,164,93,94,199,Female,Secondary,Retired,1
4,France,73,21.8,296,91,97,Never,29.1,Moderate,1,0,6,8.4,5.0,52,108,74,226,157,Female,Primary,Employed,1


In [ ]:
df_hypertension.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 174982 entries, 0 to 174981
Data columns (total 23 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Country                  174982 non-null  object 
 1   Age                      174982 non-null  int64  
 2   BMI                      174982 non-null  float64
 3   Cholesterol              174982 non-null  int64  
 4   Systolic_BP              174982 non-null  int64  
 5   Diastolic_BP             174982 non-null  int64  
 6   Smoking_Status           174982 non-null  object 
 7   Alcohol_Intake           174982 non-null  float64
 8   Physical_Activity_Level  174982 non-null  object 
 9   Family_History           174982 non-null  int64  
 10  Diabetes                 174982 non-null  int64  
 11  Stress_Level             174982 non-null  int64  
 12  Salt_Intake              174982 non-null  float64
 13  Sleep_Duration           174982 non-null  float64
 14  Hear

In [ ]:
# Encode
categorical_cols = ["Smoking_Status", "Physical_Activity_Level", 
                    "Gender", "Education_Level", "Employment_Status"]

# One-hot encode, drop_first avoids dummy trap
df_hypertension_encoded = pd.get_dummies(
    df_hypertension, 
    columns=categorical_cols, 
    drop_first=True
)

# Convert any bools to int (0/1)
for col in df_hypertension_encoded.select_dtypes(include="bool").columns:
    df_hypertension_encoded[col] = df_hypertension_encoded[col].astype(int)

# Quick check
df_hypertension_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 174982 entries, 0 to 174981
Data columns (total 27 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   Country                           174982 non-null  object 
 1   Age                               174982 non-null  int64  
 2   BMI                               174982 non-null  float64
 3   Cholesterol                       174982 non-null  int64  
 4   Systolic_BP                       174982 non-null  int64  
 5   Diastolic_BP                      174982 non-null  int64  
 6   Alcohol_Intake                    174982 non-null  float64
 7   Family_History                    174982 non-null  int64  
 8   Diabetes                          174982 non-null  int64  
 9   Stress_Level                      174982 non-null  int64  
 10  Salt_Intake                       174982 non-null  float64
 11  Sleep_Duration                    174982 non-null  f

### Hypertension: Define Target and Predictors

In [ ]:
# Target
hypertension_target_Y = df_hypertension_encoded["Systolic_BP"]

# Predictors
hypertension_predictors = [
    "Age", "BMI", "Cholesterol", "Stress_Level", "Sleep_Duration",
    "Diabetes", "Family_History",
    "Smoking_Status_Former", "Smoking_Status_Never",
    "Employment_Status_Retired", "Employment_Status_Unemployed"
]

# Slice the DataFrame to get the actual data
X = df_hypertension_encoded[hypertension_predictors]

# Add constant
X = sm.add_constant(X)

### Hypertension: Linear Regression

In [ ]:
# Fit model
hypertension_results = sm.OLS(hypertension_target_Y, X).fit()

# View coefficients
hypertension_results.params

const                           135.178539
Age                              -0.002078
BMI                              -0.007095
Cholesterol                      -0.001775
Stress_Level                      0.014561
Sleep_Duration                    0.004380
Diabetes                          0.166102
Family_History                   -0.175370
Smoking_Status_Former             0.129041
Smoking_Status_Never              0.173599
Employment_Status_Retired        -0.146368
Employment_Status_Unemployed     -0.358803
dtype: float64

In [ ]:
hypertension_results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            Systolic_BP   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.260
Date:                Sun, 14 Sep 2025   Prob (F-statistic):              0.241
Time:                        14:16:30   Log-Likelihood:            -8.1852e+05
No. Observations:              174982   AIC:                         1.637e+06
Df Residuals:                  174970   BIC:                         1.637e+06
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
================================================================================================
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const                          135.1785      0.539    250.568      0.000     134.121     136.236
Age                             -0.0021      0.003     -0.695      0.487      -0.008       0.004
BMI                             -0.0071      0.009     -0.823      0.411      -0.024       0.010
Cholesterol                     -0.0018      0.001     -1.238      0.216      -0.005       0.001
Stress_Level                     0.0146      0.024      0.604      0.546      -0.033       0.062
Sleep_Duration                   0.0044      0.036      0.122      0.903      -0.066       0.075
Diabetes                         0.1661      0.124      1.335      0.182      -0.078       0.410
Family_History                  -0.1754      0.124     -1.410      0.159      -0.419       0.068
Smoking_Status_Former            0.1290      0.152      0.848      0.396      -0.169       0.427
Smoking_Status_Never             0.1736      0.152      1.138      0.255      -0.125       0.472
Employment_Status_Retired       -0.1464      0.152     -0.962      0.336      -0.444       0.152
Employment_Status_Unemployed    -0.3588      0.153     -2.352      0.019      -0.658      -0.060
==============================================================================
Omnibus:                   161992.888   Durbin-Watson:                   2.006
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            10577.646
Skew:                          -0.001   Prob(JB):                         0.00
Kurtosis:                       1.796   Cond. No.                     2.06e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.06e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

### Hypertension Dataset: Linear Regression - Polynomial Terms

In [ ]:
# Create a squared term for Age
df_hypertension_encoded["Age_sq"] = df_hypertension_encoded["Age"] ** 2

# Predictors
hypertension_predictors_sq = [
    "Age", "Age_sq", "BMI", "Cholesterol", "Stress_Level", "Sleep_Duration",
    "Diabetes", "Family_History",
    "Smoking_Status_Former", "Smoking_Status_Never",
    "Employment_Status_Retired", "Employment_Status_Unemployed"
]

# Slice the DataFrame to get the actual data
X_sq = df_hypertension_encoded[hypertension_predictors_sq]

# Add constant
X_sq = sm.add_constant(X_sq)

hypertension_target_Y = df_hypertension_encoded["Hypertension"]

In [ ]:
# Fit model
sq_results = sm.OLS(hypertension_target_Y, X_sq).fit()

# View coefficients
sq_results.params

const                           0.730183
Age                             0.000129
Age_sq                         -0.000002
BMI                            -0.000063
Cholesterol                    -0.000042
Stress_Level                   -0.000087
Sleep_Duration                 -0.000040
Diabetes                        0.001516
Family_History                  0.000743
Smoking_Status_Former          -0.000459
Smoking_Status_Never            0.002652
Employment_Status_Retired      -0.002384
Employment_Status_Unemployed   -0.003687
dtype: float64

In [ ]:
sq_results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:           Hypertension   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.8111
Date:                Sun, 14 Sep 2025   Prob (F-statistic):              0.639
Time:                        14:16:31   Log-Likelihood:            -1.0839e+05
No. Observations:              174982   AIC:                         2.168e+05
Df Residuals:                  174969   BIC:                         2.169e+05
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
================================================================================================
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const                            0.7302      0.012     63.441      0.000       0.708       0.753
Age                              0.0001      0.000      0.426      0.670      -0.000       0.001
Age_sq                       -1.878e-06   2.78e-06     -0.676      0.499   -7.32e-06    3.57e-06
BMI                          -6.288e-05      0.000     -0.422      0.673      -0.000       0.000
Cholesterol                  -4.206e-05   2.48e-05     -1.697      0.090   -9.06e-05    6.52e-06
Stress_Level                 -8.681e-05      0.000     -0.209      0.835      -0.001       0.001
Sleep_Duration               -3.964e-05      0.001     -0.064      0.949      -0.001       0.001
Diabetes                         0.0015      0.002      0.705      0.481      -0.003       0.006
Family_History                   0.0007      0.002      0.346      0.729      -0.003       0.005
Smoking_Status_Former           -0.0005      0.003     -0.175      0.861      -0.006       0.005
Smoking_Status_Never             0.0027      0.003      1.007      0.314      -0.003       0.008
Employment_Status_Retired       -0.0024      0.003     -0.907      0.364      -0.008       0.003
Employment_Status_Unemployed    -0.0037      0.003     -1.399      0.162      -0.009       0.001
==============================================================================
Omnibus:                    73818.878   Durbin-Watson:                   1.994
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            35704.599
Skew:                          -0.973   Prob(JB):                         0.00
Kurtosis:                       1.948   Cond. No.                     4.29e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 4.29e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

### Hypertension Dataset: Linear Regression - Interaction Terms

In [ ]:
# Create an interaction term between Age and PhysHlth
df_hypertension_encoded["Age_BMI"] = df_hypertension_encoded["Age"] * df_hypertension_encoded["BMI"]

# Predictors
hypertension_predictors_int = [
    "Age", "Age_sq", "BMI", "Cholesterol", "Stress_Level", "Sleep_Duration",
    "Diabetes", "Family_History",
    "Smoking_Status_Former", "Smoking_Status_Never",
    "Employment_Status_Retired", "Employment_Status_Unemployed"
]

# Slice the DataFrame to get the actual data
X_hypertension_int = df_hypertension_encoded[hypertension_predictors_int]

# Add constant
X_hypertension_int = sm.add_constant(X_hypertension_int)

In [ ]:
# Fit model
int_results = sm.OLS(hypertension_target_Y, X_hypertension_int).fit()  

# View coefficients
int_results.params

const                           0.730183
Age                             0.000129
Age_sq                         -0.000002
BMI                            -0.000063
Cholesterol                    -0.000042
Stress_Level                   -0.000087
Sleep_Duration                 -0.000040
Diabetes                        0.001516
Family_History                  0.000743
Smoking_Status_Former          -0.000459
Smoking_Status_Never            0.002652
Employment_Status_Retired      -0.002384
Employment_Status_Unemployed   -0.003687
dtype: float64

In [ ]:
int_results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:           Hypertension   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.8111
Date:                Sun, 14 Sep 2025   Prob (F-statistic):              0.639
Time:                        14:16:31   Log-Likelihood:            -1.0839e+05
No. Observations:              174982   AIC:                         2.168e+05
Df Residuals:                  174969   BIC:                         2.169e+05
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
================================================================================================
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const                            0.7302      0.012     63.441      0.000       0.708       0.753
Age                              0.0001      0.000      0.426      0.670      -0.000       0.001
Age_sq                       -1.878e-06   2.78e-06     -0.676      0.499   -7.32e-06    3.57e-06
BMI                          -6.288e-05      0.000     -0.422      0.673      -0.000       0.000
Cholesterol                  -4.206e-05   2.48e-05     -1.697      0.090   -9.06e-05    6.52e-06
Stress_Level                 -8.681e-05      0.000     -0.209      0.835      -0.001       0.001
Sleep_Duration               -3.964e-05      0.001     -0.064      0.949      -0.001       0.001
Diabetes                         0.0015      0.002      0.705      0.481      -0.003       0.006
Family_History                   0.0007      0.002      0.346      0.729      -0.003       0.005
Smoking_Status_Former           -0.0005      0.003     -0.175      0.861      -0.006       0.005
Smoking_Status_Never             0.0027      0.003      1.007      0.314      -0.003       0.008
Employment_Status_Retired       -0.0024      0.003     -0.907      0.364      -0.008       0.003
Employment_Status_Unemployed    -0.0037      0.003     -1.399      0.162      -0.009       0.001
==============================================================================
Omnibus:                    73818.878   Durbin-Watson:                   1.994
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            35704.599
Skew:                          -0.973   Prob(JB):                         0.00
Kurtosis:                       1.948   Cond. No.                     4.29e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 4.29e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

### Hypertension Dataset: Linear Regression – Multicollinearity (VIF)

In [ ]:
# Use the same predictors from your interaction model (before adding the constant)
X_vif = df_hypertension_encoded[hypertension_predictors_int].dropna()

# Add constant to match regression setup
X_vif_const = sm.add_constant(X_vif)

# Compute VIF for each column
vif_data = pd.DataFrame()
vif_data["feature"] = X_vif_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif_const.values, i)
                   for i in range(X_vif_const.shape[1])]

vif_data

,feature,VIF
0,const,114.685766
1,Age,34.048937
2,Age_sq,34.048963
3,BMI,1.000029
4,Cholesterol,1.000103
5,Stress_Level,1.000040
6,Sleep_Duration,1.000009
7,Diabetes,1.000044
8,Family_History,1.000021
9,Smoking_Status_Former,1.330592


### Week 1 Conclusion — Hypertension Dataset
For the hypertension dataset, polynomial and interaction terms were introduced to explore how demographic, lifestyle, and health behavior variables jointly relate to hypertension outcomes. The inclusion of encoded categorical predictors such as smoking status and physical activity provided a broader understanding of how lifestyle combinations affect blood pressure patterns.  

Model evaluation indicated that interaction terms modestly improved the model’s explanatory power, capturing nuanced effects between age, cholesterol, and physical activity level. However, higher-order polynomial terms offered diminishing returns and increased the potential for overfitting. This was managed by keeping the polynomial degree low and checking for inflated VIF values, which confirmed the absence of severe multicollinearity.  

These results align with the exploratory findings that no single factor drives hypertension alone; rather, it emerges from interdependent behavioral and biological variables. The polynomial model strengthened that interpretation while maintaining generalizability.


### Week 1 Summary — Polynomial and Interaction Modeling
Across the three datasets—diabetes, hypertension, and kidney—the application of polynomial and interaction terms provided valuable insight into how complex, nonlinear relationships can enhance model accuracy while still remaining interpretable. Introducing higher-order features allowed subtle patterns to emerge, such as curvilinear effects between health status indicators and outcomes like BMI, blood pressure, and GFR.  

Model comparisons demonstrated that small polynomial degrees improved explanatory power without causing substantial overfitting. Regular monitoring of the Variance Inflation Factor (VIF) confirmed that multicollinearity remained within acceptable bounds. Cross-validation and residual analysis further ensured that each model balanced flexibility with generalizability.  

Overall, Week 1 established the foundation for more advanced modeling in later weeks by reinforcing key regression principles: understanding bias-variance trade-offs, recognizing the role of feature interactions, and maintaining interpretability as model complexity increases. These insights guide how I will approach subsequent techniques, such as regularization, feature selection, and ensemble methods, in the following milestones.
